In [ ]:
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score,precision_score, recall_score, f1_score
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import numpy as np
from nltk.stem import PorterStemmer
from sklearn.model_selection import GridSearchCV
from scipy import sparse as sp
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

In [29]:
data = pd.read_csv("E:\\AI-Powered Customer Query Classifier\\customer_query_dataset_humanized_12924.csv")
df = data.copy()

In [30]:
data['Priority'].value_counts()

Priority
Low       5746
Medium    4556
High      2622
Name: count, dtype: int64

In [31]:
df.head()

,Customer_query,Category,Intent,Priority
0,I need help with this as soon as possible: the...,Pricing,Billing Request,High
1,I'm having trouble with the price of the busin...,Pricing,Pricing Information,Medium
2,I urgently need help with changing my service.,Service Inquiry,Service Request,High
3,Could someone help me with what I'd pay for th...,Pricing,Pricing Information,Medium
4,I urgently need help with your service coverage.,Service Inquiry,Service Information,High


In [32]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12924 entries, 0 to 12923
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   Customer_query  12924 non-null  str  
 1   Category        12924 non-null  str  
 2   Intent          12924 non-null  str  
 3   Priority        12924 non-null  str  
dtypes: str(4)
memory usage: 1.7 MB


In [33]:
def convert_lowercase(text):
    return text.lower()
df['Customer_query'] = df['Customer_query'].apply(convert_lowercase)

In [34]:
def remove_punctuation(text):
    return ''.join(char for char in text if char.isalnum() or char.isspace())
df["Customer_query"] = df["Customer_query"].apply(remove_punctuation)

In [35]:
def stopwords_removal(text):
    stop_words = set(stopwords.words('english'))
    tokens = word_tokenize(text)
    filtered_tokens = [token for token in tokens if token not in stop_words]
    return ' '.join(filtered_tokens)

df['Customer_query'] = df['Customer_query'].apply(stopwords_removal)

In [36]:
df['Customer_query'].sample(5)

8181                     im trying find price current plan
6077     need information service information account r...
12657    need information pricing information current p...
5230                            need help know hr platform
3077     need information service request hr platform r...
Name: Customer_query, dtype: str

In [37]:
def convert_stemming(text):
    stemmer = PorterStemmer()
    tokens = word_tokenize(text)
    stemmed_tokens = [stemmer.stem(token) for token in tokens]
    return ' '.join(stemmed_tokens)

df['Customer_query'] = df['Customer_query'].apply(convert_stemming)

In [38]:
def tokenization(text):
    return word_tokenize(text)
df['Customer_query'] = df['Customer_query'].apply(tokenization)


In [ ]:

vectorizer = TfidfVectorizer(max_df=0.9, min_df=5, max_features=3000, ngram_range=(1,2))
df['Customer_query_text'] = df['Customer_query'].apply(lambda tokens: ' '.join(tokens) if isinstance(tokens, (list, tuple)) else str(tokens))

# Category Model

In [55]:
x_train, x_test, y_train, y_test = train_test_split(df['Customer_query_text'], df['Category'], test_size=0.2, random_state=42)

In [70]:
mnb = MultinomialNB()

In [ ]:
y_train_arr = np.asarray(y_train).ravel()

pipeline = Pipeline([('vect', vectorizer), ('clf', MultinomialNB(alpha=1.0))])
param_grid = {
    'clf__alpha': [0.01, 0.1, 0.5, 1.0, 2.0],
    'clf__fit_prior': [True, False],
    'clf__class_prior': [None]
}
grid = GridSearchCV(pipeline, param_grid=param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid.fit(x_train, y_train_arr)

model_priority = grid.best_estimator_
print('Best params:', grid.best_params_, 'CV score:', grid.best_score_)

X_train = model_priority.named_steps['vect'].transform(x_train)
X_test = model_priority.named_steps['vect'].transform(x_test)

Best params: {'clf__alpha': 0.01, 'clf__class_prior': None, 'clf__fit_prior': True} CV score: 0.9852982157679067


In [59]:
pickle.dump(vectorizer, open('category_tfidf_vectorizer.pkl', 'wb'))
pickle.dump(mnb, open('category_classifier.pkl', 'wb'))

In [ ]:
y_train_arr = np.asarray(y_train).ravel()
y_test_arr = np.asarray(y_test).ravel()

if hasattr(model_priority, 'predict'):
    y_pred_train = model_priority.predict(x_train)
    y_pred_test = model_priority.predict(x_test)
else:
    y_pred_train = model_priority.predict(X_train)
    y_pred_test = model_priority.predict(X_test)

# accuracy
acc_train = accuracy_score(y_train_arr, y_pred_train)
acc_test = accuracy_score(y_test_arr, y_pred_test)
print(f"Train accuracy: {acc_train:.4f}")
print(f"Test accuracy:  {acc_test:.4f}")


try:
    print("Model.score (test):", model_priority.score(x_test if hasattr(model_priority, 'named_steps') else X_test, y_test_arr))
except Exception:
    print("Model.score (test):", model_priority.score(X_test, y_test_arr))

print('\nClassification report (test):')
print(classification_report(y_test_arr, y_pred_test))
print('Confusion matrix (test):')
print(confusion_matrix(y_test_arr, y_pred_test))

Train accuracy: 0.9879
Test accuracy:  0.9876
Model.score (test): 0.9876208897485493

Classification report (test):
                   precision    recall  f1-score   support

        Complaint       1.00      1.00      1.00       420
  General Inquiry       1.00      0.97      0.98       412
          Pricing       1.00      1.00      1.00       433
  Product Inquiry       0.99      0.96      0.98       445
  Service Inquiry       1.00      1.00      1.00       438
Technical Support       0.94      1.00      0.97       437

         accuracy                           0.99      2585
        macro avg       0.99      0.99      0.99      2585
     weighted avg       0.99      0.99      0.99      2585

Confusion matrix (test):
[[419   0   0   0   0   1]
 [  0 398   1   4   0   9]
 [  0   0 433   0   0   0]
 [  0   0   0 428   0  17]
 [  0   0   0   0 438   0]
 [  0   0   0   0   0 437]]


# Intent Model

In [61]:
x_train, x_test, y_train, y_test = train_test_split(df['Customer_query_text'], df['Intent'], test_size=0.2, random_state=42)

In [ ]:
y_train_arr = np.asarray(y_train).ravel()

pipeline = Pipeline([('vect', vectorizer), ('clf', MultinomialNB(alpha=1.0))])
param_grid = {
    'clf__alpha': [0.01, 0.1, 0.5, 1.0, 2.0],
    'clf__fit_prior': [True, False],
    'clf__class_prior': [None]
}
grid = GridSearchCV(pipeline, param_grid=param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid.fit(x_train, y_train_arr)

model_priority = grid.best_estimator_
print('Best params:', grid.best_params_, 'CV score:', grid.best_score_)

X_train = model_priority.named_steps['vect'].transform(x_train)
X_test = model_priority.named_steps['vect'].transform(x_test)

Best params: {'clf__alpha': 0.01, 'clf__class_prior': None, 'clf__fit_prior': False} CV score: 0.9896506678120488


In [ ]:
y_train_arr = np.asarray(y_train).ravel()
y_test_arr = np.asarray(y_test).ravel()

if hasattr(model_priority, 'predict'):
    y_pred_train = model_priority.predict(x_train)
    y_pred_test = model_priority.predict(x_test)
else:
    y_pred_train = model_priority.predict(X_train)
    y_pred_test = model_priority.predict(X_test)

acc_train = accuracy_score(y_train_arr, y_pred_train)
acc_test = accuracy_score(y_test_arr, y_pred_test)
print(f"Train accuracy: {acc_train:.4f}")
print(f"Test accuracy:  {acc_test:.4f}")

try:
    print("Model.score (test):", model_priority.score(x_test if hasattr(model_priority, 'named_steps') else X_test, y_test_arr))
except Exception:
    print("Model.score (test):", model_priority.score(X_test, y_test_arr))

print('\nClassification report (test):')
print(classification_report(y_test_arr, y_pred_test))
print('Confusion matrix (test):')
print(confusion_matrix(y_test_arr, y_pred_test))

Train accuracy: 0.9951
Test accuracy:  0.9880
Model.score (test): 0.9880077369439072

Classification report (test):
                     precision    recall  f1-score   support

    Account Request       1.00      0.96      0.98       101
    Billing Request       0.73      1.00      0.85        22
 Escalation Request       1.00      1.00      1.00        21
General Information       1.00      0.95      0.98       311
Pricing Information       1.00      1.00      1.00       411
Product Information       0.98      0.98      0.98       221
    Product Request       1.00      1.00      1.00       224
  Service Complaint       1.00      0.99      0.99       399
Service Information       0.99      1.00      1.00       116
    Service Request       1.00      0.99      1.00       322
    Support Request       0.57      1.00      0.72        17
    Technical Issue       0.99      1.00      1.00       420

           accuracy                           0.99      2585
          macro avg       0.

In [65]:
pickle.dump(vectorizer, open('Intent_tfidf_vectorizer.pkl', 'wb'))
pickle.dump(mnb, open('Intent_classifier.pkl', 'wb'))

In [ ]:
new_query = ["When will the next milestone deliverable be ready for review?"]

predicted_label = model_priority.predict(new_query)
print("Predicted priority:", predicted_label[0])

if hasattr(model_priority, "predict_proba"):
    proba = model_priority.predict_proba(new_query)
    print("Prediction probabilities:", proba[0])

Predicted priority: Escalation Request
Prediction probabilities: [2.24045300e-04 1.15520545e-03 9.96752479e-01 7.44783731e-05
 5.90745341e-05 1.25472136e-04 1.11940358e-04 5.74055317e-05
 1.87450583e-04 8.15719924e-05 1.10750268e-03 6.33744900e-05]


# Priority Model

In [67]:
x_train, x_test, y_train, y_test = train_test_split(df['Customer_query_text'], df['Priority'], test_size=0.2, random_state=42)

In [ ]:
y_train_arr = np.asarray(y_train).ravel()

pipeline = Pipeline([('vect', vectorizer), ('clf', MultinomialNB(alpha=1.0))])
param_grid = {
    'clf__alpha': [0.01, 0.1, 0.5, 1.0, 2.0],
    'clf__fit_prior': [True, False],
    'clf__class_prior': [None]
}
grid = GridSearchCV(pipeline, param_grid=param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid.fit(x_train, y_train_arr)

model_priority = grid.best_estimator_
print('Best params:', grid.best_params_, 'CV score:', grid.best_score_)

X_train = model_priority.named_steps['vect'].transform(x_train)
X_test = model_priority.named_steps['vect'].transform(x_test)

Best params: {'clf__alpha': 2.0, 'clf__class_prior': None, 'clf__fit_prior': True} CV score: 0.7363378091198245


In [ ]:
y_train_arr = np.asarray(y_train).ravel()
y_test_arr = np.asarray(y_test).ravel()

if hasattr(model_priority, 'predict'):
    y_pred_train = model_priority.predict(x_train)
    y_pred_test = model_priority.predict(x_test)
else:
    y_pred_train = model_priority.predict(X_train)
    y_pred_test = model_priority.predict(X_test)

acc_train = accuracy_score(y_train_arr, y_pred_train)
acc_test = accuracy_score(y_test_arr, y_pred_test)
print(f"Train accuracy: {acc_train:.4f}")
print(f"Test accuracy:  {acc_test:.4f}")

try:
    print("Model.score (test):", model_priority.score(x_test if hasattr(model_priority, 'named_steps') else X_test, y_test_arr))
except Exception:
    print("Model.score (test):", model_priority.score(X_test, y_test_arr))

print('\nClassification report (test):')
print(classification_report(y_test_arr, y_pred_test))
print('Confusion matrix (test):')
print(confusion_matrix(y_test_arr, y_pred_test))

Train accuracy: 0.7421
Test accuracy:  0.7377
Model.score (test): 0.7377176015473887

Classification report (test):
              precision    recall  f1-score   support

        High       0.91      0.56      0.69       544
         Low       0.64      0.98      0.78      1128
      Medium       0.94      0.55      0.69       913

    accuracy                           0.74      2585
   macro avg       0.83      0.70      0.72      2585
weighted avg       0.80      0.74      0.73      2585

Confusion matrix (test):
[[ 305  230    9]
 [   0 1103   25]
 [  29  385  499]]


In [ ]:
pickle.dump(vectorizer, open('Intent_tfidf_vectorizer.pkl', 'wb'))
pickle.dump(mnb, open('Intent_classifier.pkl', 'wb'))